# Multi-Scale SR — Kaggle GPU Training

Pull the `multiscale_sr` code from GitHub, read the CMS jet parquet data from an attached Kaggle Dataset, train a chosen scale on the **full dataset** on the Kaggle GPU, evaluate tagging efficiency, and persist everything under `/kaggle/working` for download.

## Before you Run-All (prerequisites)

- **(P1) Push your latest code to GitHub first.** This notebook pulls code from git — any uncommitted local work will *not* be here. Commit + push the branch you want, then set `REPO_BRANCH` in Cell 2 to match.
- **(P2) Attach the data as a Kaggle Dataset.** Upload the `*.parquet` jet files as a Kaggle Dataset and attach it (right panel → *Add Input*). It mounts read-only at `/kaggle/input/<slug>/`. Data does **not** come from git.
- **(P3) Notebook settings:** *Accelerator = GPU* (T4 x2 or P100) and *Internet = ON* (needed for `git clone` + `pip`). Limits: ~12 h/session, ~30 h/week GPU, ~19 GB writable `/kaggle/working`.
- **(P4) Optional W&B:** add `WANDB_API_KEY` via *Add-ons → Secrets*. If present it logs to W&B; otherwise it falls back to `--no-wandb`.

## How to run
1. Set `SCALE` and `EPOCHS` in **Cell 6** (default 16×, 30 epochs).
2. *Run All*.
3. Download results from the **Output** tab (or the `.zip` created in the last cell).


## Cell 1 — Environment check (GPU is required)

In [ ]:
import subprocess, sys
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Enable it: notebook right panel -> Accelerator -> GPU (T4 x2 / P100), "
        "then Run All again."
    )
print("device:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## Cell 2 — Get the code (clone repo at a pinned branch)
Re-running is safe: the clone dir is removed first. Set `REPO_BRANCH` to whatever you pushed in (P1).

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/rajveer43/cms-superres-reconstruction.git"
REPO_BRANCH = "master"   # <- match the branch you pushed your code to
CLONE_DIR   = Path("/kaggle/working/repo")
CODE_DIR    = CLONE_DIR / "multiscale_sr"

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)   # idempotent re-clone

r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True, text=True,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise SystemExit(
        f"git clone failed (branch '{REPO_BRANCH}'). Check: Internet=ON, the branch exists, "
        f"and the repo is public. Fallback: wget "
        f"https://github.com/rajveer43/cms-superres-reconstruction/archive/refs/heads/{REPO_BRANCH}.tar.gz"
    )
if not CODE_DIR.exists():
    raise SystemExit(f"Expected code at {CODE_DIR} but it is missing after clone.")

os.chdir(CODE_DIR)
commit = subprocess.run(["git", "-C", str(CLONE_DIR), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("cwd    :", os.getcwd())
print("commit :", commit, "(branch", REPO_BRANCH + ")")

## Cell 3 — Dependencies (install only what's missing)
Kaggle ships torch/numpy/pyarrow/sklearn/matplotlib/h5py/pyyaml. We only add `wandb` and `python-dotenv`; torch is **not** touched.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        print(f"ok: {pkg}")
    except ImportError:
        print(f"installing: {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("wandb", "wandb"), ("python-dotenv", "dotenv")]:
    ensure(pkg, imp)

import torch, numpy, pyarrow, sklearn
print("torch", torch.__version__, "| numpy", numpy.__version__,
      "| pyarrow", pyarrow.__version__, "| sklearn", sklearn.__version__)

## Cell 4 — Locate the attached data
Finds the `/kaggle/input/*` folder that contains `*.parquet` and reports row counts from parquet metadata (no decode).

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

INPUT_ROOT = Path("/kaggle/input")
candidates = sorted({p.parent for p in INPUT_ROOT.glob("*/**/*.parquet")})
if not candidates:
    raise SystemExit(
        "No *.parquet found under /kaggle/input. Attach your jet dataset: "
        "right panel -> Add Input -> your Kaggle Dataset of parquet files."
    )

DATA_DIR = candidates[0]
if len(candidates) > 1:
    print("Multiple parquet folders found; using the first. Edit DATA_DIR if wrong:")
    for c in candidates:
        print("  ", c)

files = sorted(DATA_DIR.glob("*.parquet"))
total = 0
print("DATA_DIR:", DATA_DIR)
for f in files:
    n = pq.ParquetFile(f).metadata.num_rows
    total += n
    print(f"  {f.name}: {n:,} rows")
print(f"TOTAL: {total:,} jets across {len(files)} file(s)")
print("(code splits by file: first files -> train, last -> val)")

## Cell 5 — W&B (optional, via Kaggle Secrets)
Reads `WANDB_API_KEY` from Kaggle Secrets if you added one. No key → training runs with `--no-wandb`. Never hardcode a key.

In [ ]:
import os

USE_WANDB = False
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        USE_WANDB = True
        print("W&B: key found -> logging ENABLED")
except Exception as e:
    print("W&B: no secret found ->", type(e).__name__)

if not USE_WANDB:
    print("W&B: DISABLED (training will use --no-wandb). "
          "To enable: Add-ons -> Secrets -> WANDB_API_KEY.")

## Cell 6 — Parameters (edit these)
Full dataset per epoch (no batch cap). On a T4 an ~84k-sample epoch is a few minutes, so 30 epochs fits inside one 12 h session. To continue a longer run across sessions, set `RESUME` (Cell 10).

In [ ]:
SCALE            = 16                 # 16 / 32 / 64
EPOCHS           = 30
RUN_NAME         = f"kaggle_{SCALE}x_full"
EXPERIMENTS_ROOT = "/kaggle/working/experiments"   # persisted as notebook output
SEED             = 42
RESUME           = None              # e.g. ".../checkpoints/latest.pt" to continue

CONFIG = f"configs/scale_{SCALE}.yaml"
assert Path(CONFIG).exists(), f"missing {CONFIG} in {os.getcwd()} — check the clone (Cell 2)"
print(f"scale={SCALE}  epochs={EPOCHS}  run={RUN_NAME}  wandb={USE_WANDB}")
print(f"config={CONFIG}  data={DATA_DIR}  experiments_root={EXPERIMENTS_ROOT}")

## Cell 7 — Train on the FULL dataset (streamed logs)
No `--max-train-batches` → every epoch sees the whole training split. `env.py` auto-selects CUDA (num_workers=4, AMP, float16); the resolved config line is printed at the top of the log.

In [ ]:
import json, subprocess, sys
from pathlib import Path

cmd = [
    sys.executable, "train.py",
    "--config", CONFIG,
    "--data-dir", str(DATA_DIR),
    "--scale", str(SCALE),
    "--epochs", str(EPOCHS),
    "--run-name", RUN_NAME,
    "--experiments-root", EXPERIMENTS_ROOT,
]
if RESUME:
    cmd += ["--resume", RESUME]
if not USE_WANDB:
    cmd += ["--no-wandb"]
print("RUN:", " ".join(cmd), "\n", flush=True)

# Stream output line-by-line so progress shows live.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"training failed (exit {proc.returncode}) — see the log above.")

# Locate the newest run dir and report final metrics.
run_dirs = sorted(Path(EXPERIMENTS_ROOT).glob(f"*{SCALE}x*{RUN_NAME}*"),
                  key=lambda p: p.stat().st_mtime)
RUN_DIR = run_dirs[-1]
print("\nRUN_DIR:", RUN_DIR)
last = [l for l in (RUN_DIR / "metrics.jsonl").read_text().splitlines() if l.strip()][-1]
m = json.loads(last)
print(f"epoch={m.get('epoch')}  val_l1={m.get('val_l1'):.4f}  energy_response={m.get('val_energy_response'):.4f}")

## Cell 8 — Evaluate tagging efficiency on the trained checkpoint

In [ ]:
import json, subprocess, sys

CKPT        = RUN_DIR / "checkpoints" / "best.pt"
CLASSIF_DIR = RUN_DIR / "figures" / "classification"
assert CKPT.exists(), f"no checkpoint at {CKPT}"

cmd = [
    sys.executable, "classification_eval.py",
    "--checkpoint", str(CKPT),
    "--data-dir", str(DATA_DIR),
    "--out-dir", str(CLASSIF_DIR),
    "--seed", str(SEED),
]
print("RUN:", " ".join(cmd), "\n", flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"eval failed (exit {proc.returncode}) — see log above.")

res = json.loads((CLASSIF_DIR / "classification_eval.json").read_text())
p = res["primary_fixed_hr_tagger"]
print("\n===== HEADLINE =====")
print(f"AUC  HR={p['auc']['hr']:.3f}  LR={p['auc']['lr']:.3f}  SR={p['auc']['sr']:.3f}")
print(f"tagging efficiency (AUC_SR/AUC_HR) = {p['tagging_efficiency_sr_over_hr']*100:.1f}%")
print(f"recovery (LR->HR gap closed)       = {p['recovery_fraction_lr_to_hr']*100:.1f}%")

## Cell 9 — Show key figures + zip outputs for download

In [ ]:
import shutil
from pathlib import Path
from IPython.display import Image, display

for name in ["auc_summary_bar.png", "roc_overlay.png"]:
    fp = CLASSIF_DIR / name
    if fp.exists():
        print(name)
        display(Image(filename=str(fp)))

samples = sorted(RUN_DIR.glob("figures/sample_epoch_*.png"))
if samples:
    print(samples[-1].name)
    display(Image(filename=str(samples[-1])))

# RUN_DIR is under /kaggle/working -> already saved as notebook output.
zip_base = str(Path("/kaggle/working") / RUN_DIR.name)
shutil.make_archive(zip_base, "zip", root_dir=str(RUN_DIR))
print("\nDownloadable archive:", zip_base + ".zip")
print("Run dir (persisted):  ", RUN_DIR)

## Cell 10 — After the run

**Download results**
- Everything under `/kaggle/working` (the `experiments/...` run dir + the `.zip`) is saved to the notebook's **Output** tab. Open it and download.

**Resume a longer run in a new session** (Kaggle caps sessions at ~12 h)
1. Re-run Cells 1–6.
2. In Cell 6 set `RESUME` to the persisted checkpoint from the previous session, e.g.
   `RESUME = "/kaggle/input/<prev-output-dataset>/experiments/<run>/checkpoints/latest.pt"`
   (attach the previous run's Output as an input dataset), and bump `EPOCHS`.
3. Re-run Cell 7 onward.

**Train another scale**
- Start a fresh session, set `SCALE = 32` (or `64`) in Cell 6, and *Run All*. 64× is already solved (~100% efficiency); 16× and 32× are the ones that benefit most from full-dataset GPU training.
